<a href="https://colab.research.google.com/github/ekc2024/ScamGuard-MY/blob/main/Credit_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experian-Style Financial Document Analyzer (PDPA Compliant)
This notebook extracts unstructured text from financial documents — **credit reports, invoices, and purchase orders** — in PDF or spreadsheet form, redacts Personally Identifiable Information (PII) like NRICs, emails, and phone numbers for **PDPA compliance**, auto-detects the document type, pulls out structured fields relevant to that type, flags risks, and presents an AI-powered dashboard.

**Supports:** PDF and spreadsheet (`.xlsx` / `.csv`) input. You can generate mock samples to test with, or upload your own files.

### 1. Install Dependencies

In [ ]:
!pip install pdfplumber openpyxl reportlab -q

### 2. Core Engine — PII Masking, Extraction, Document Type Detection, Structured Field Extraction, Risk Detection
Run this cell once. Everything the pipeline needs lives here.

In [ ]:
import re
import pdfplumber
import pandas as pd


# ---------------------------------------------------------------------------
# 2.1 PII Masking
# ---------------------------------------------------------------------------
def mask_experian_pii(text: str) -> str:
    """Masks Malaysian NRICs, Emails, and Phone Numbers for PDPA Compliance."""
    nric_pattern = r"\b\d{6}-\d{2}-\d{4}\b"
    email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"
    phone_pattern = r"\b01[0-9][-\s]?\d{3,4}[-\s]?\d{4}\b"

    masked = re.sub(nric_pattern, "[REDACTED_NRIC]", text)
    masked = re.sub(email_pattern, "[REDACTED_EMAIL]", masked)
    masked = re.sub(phone_pattern, "[REDACTED_PHONE]", masked)
    return masked


# ---------------------------------------------------------------------------
# 2.2 Extraction (PDF or spreadsheet) -> sanitized text
# ---------------------------------------------------------------------------
def extract_from_spreadsheet(file_path: str) -> str:
    """Extracts and flattens spreadsheet data (xlsx or csv) into text for the same pipeline."""
    if file_path.lower().endswith(".csv"):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path)

    lines = []
    for _, row in df.iterrows():
        line = " | ".join(f"{col}: {val}" for col, val in row.items())
        lines.append(line)
    return "\n".join(lines)


def extract_any(file_path: str) -> str:
    """Router: sends the file to the right extractor based on its extension, then masks PII."""
    if file_path.lower().endswith((".xlsx", ".xls", ".csv")):
        raw_text = extract_from_spreadsheet(file_path)
    else:
        raw_text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    raw_text += text + "\n"
    return mask_experian_pii(raw_text)


# ---------------------------------------------------------------------------
# 2.3 Document type detection
# ---------------------------------------------------------------------------
def detect_document_type(sanitized_text: str) -> str:
    """
    Classifies the document as 'invoice', 'purchase_order', 'credit_report', or 'unknown'
    based on distinctive header/keyword signals. Checked in order of specificity.
    """
    text_lower = sanitized_text.lower()

    if "purchase order" in text_lower or re.search(r"\bpo number\b", text_lower) or re.search(r"\bpo[\s#\-]*\d", text_lower):
        return "purchase_order"
    if "tax invoice" in text_lower or "invoice no" in text_lower or "invoice number" in text_lower or re.search(r"\binvoice\b", text_lower):
        return "invoice"
    if "credit assessment" in text_lower or "credit report" in text_lower or "experian" in text_lower:
        return "credit_report"
    return "unknown"


# ---------------------------------------------------------------------------
# 2.4 Structured field extraction, per document type
# ---------------------------------------------------------------------------
def _first_match(pattern, text, group=1, flags=re.IGNORECASE):
    m = re.search(pattern, text, flags)
    return m.group(group).strip() if m else None


def extract_invoice_fields(sanitized_text: str) -> dict:
    return {
        "invoice_no": _first_match(r"invoice\s*(?:no|number)\.?:?\s*([A-Za-z0-9\-]+)", sanitized_text),
        "invoice_date": _first_match(r"invoice\s*date:?\s*([\d/\-]+)", sanitized_text),
        "due_date": _first_match(r"due\s*date:?\s*([\d/\-]+)", sanitized_text),
        "bill_to": _first_match(r"bill\s*to:?\s*(.+)", sanitized_text),
        "total_due": _first_match(r"total\s*due:?\s*(RM[\s\d,\.]+)", sanitized_text),
        "payment_status": _first_match(r"payment\s*status:?\s*([A-Za-z ]+)", sanitized_text),
    }


def extract_po_fields(sanitized_text: str) -> dict:
    return {
        "po_number": _first_match(r"po\s*number:?\s*([A-Za-z0-9\-]+)", sanitized_text),
        "po_date": _first_match(r"po\s*date:?\s*([\d/\-]+)", sanitized_text),
        "expected_delivery": _first_match(r"expected\s*delivery:?\s*([\d/\-]+)", sanitized_text),
        "vendor": _first_match(r"vendor:?\s*(.+)", sanitized_text),
        "total_po_value": _first_match(r"total\s*po\s*value:?\s*(RM[\s\d,\.]+)", sanitized_text),
        "payment_terms": _first_match(r"payment\s*terms:?\s*([A-Za-z0-9 ]+)", sanitized_text),
    }


def extract_credit_report_fields(sanitized_text: str) -> dict:
    return {
        "company_name": _first_match(r"company\s*name:?\s*(.+)", sanitized_text),
        "director_name": _first_match(r"director\s*name:?\s*(.+)", sanitized_text),
        "total_liabilities": _first_match(r"(?:total\s*outstanding\s*)?liabilities[^\d]*(RM[\s\d,\.]+)", sanitized_text),
    }


def extract_structured_fields(sanitized_text: str, doc_type: str) -> dict:
    """Dispatches to the right field extractor based on detected document type."""
    if doc_type == "invoice":
        return extract_invoice_fields(sanitized_text)
    if doc_type == "purchase_order":
        return extract_po_fields(sanitized_text)
    if doc_type == "credit_report":
        return extract_credit_report_fields(sanitized_text)
    return {}


# ---------------------------------------------------------------------------
# 2.5 Risk / exception detection
# ---------------------------------------------------------------------------
DEFAULT_RISK_KEYWORDS = [
    "lawsuit", "liabilities", "deteriorating", "unpaid", "debt", "risk",
    "overdue", "suspension", "default", "penalty", "dispute", "terminate",
]


def detect_risks(sanitized_text: str, risk_keywords=None) -> list:
    """Flags lines containing risk-related keywords. NOTE: this is a keyword match,
    not semantic understanding — phrasing outside this list will not be caught."""
    if risk_keywords is None:
        risk_keywords = DEFAULT_RISK_KEYWORDS

    detected_risks = []
    for line in sanitized_text.split('\n'):
        if any(keyword in line.lower() for keyword in risk_keywords):
            if line.strip() and line.strip() not in detected_risks:
                detected_risks.append(line.strip())
    return detected_risks


# ---------------------------------------------------------------------------
# 2.6 Full pipeline entry point
# ---------------------------------------------------------------------------
def analyze_document(file_path: str) -> dict:
    """
    Runs the full pipeline: extract -> mask PII -> detect document type ->
    extract structured fields -> detect risks. Returns a result dict
    (rather than just printing) so it can feed the dashboard step too.
    """
    try:
        sanitized_text = extract_any(file_path)
    except FileNotFoundError:
        print(f"Error: Could not find {file_path}")
        print("Please make sure you have generated or uploaded the file first.")
        return {}

    doc_type = detect_document_type(sanitized_text)
    fields = extract_structured_fields(sanitized_text, doc_type)
    risks = detect_risks(sanitized_text)

    result = {
        "file_path": file_path,
        "document_type": doc_type,
        "sanitized_text": sanitized_text,
        "fields": fields,
        "risks": risks,
    }

    print(f"--- FILE: {file_path} ---")
    print(f"Detected Document Type: {doc_type.upper()}\n")

    print("--- PDPA SANITIZED OUTPUT ---")
    print(sanitized_text)

    print("\n--- STRUCTURED FIELDS ---")
    if fields:
        for k, v in fields.items():
            print(f"{k}: {v if v else '(not found)'}")
    else:
        print("(No structured field rules defined for this document type.)")

    print("\n--- DETECTED RISK / EXCEPTION LINES ---")
    if risks:
        for idx, r in enumerate(risks, 1):
            print(f"[{idx}] {r}")
    else:
        print("(No risk keywords matched.)")

    return result


# Kept for backward compatibility with earlier versions of this notebook
def analyze_credit_report(file_path: str):
    return analyze_document(file_path)

### 3. (Option A) Generate Mock Sample Documents
Run this cell to create three simulated PDFs to test the full pipeline against: a **credit report**, an **invoice**, and a **purchase order** — each with artificial Malaysian PII and a risk/exception signal built in.

**Skip this if you'd rather upload your own files in Step 4.**

In [5]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


def create_mock_credit_report(filename="Credit_Assessment_Pinnacle.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "EXPERIAN COMMERCIAL CREDIT ASSESSMENT")
    c.drawString(100, 730, "------------------------------------------------------------")
    c.drawString(100, 700, "Company Name: Pinnacle Tech Solutions")
    c.drawString(100, 680, "Director Name: Ahmad Razak")
    c.drawString(100, 660, "Director NRIC: 880412-14-5531")
    c.drawString(100, 640, "Contact Email: ahmad.razak@pinnacle.com.my")
    c.drawString(100, 620, "Mobile Phone: 012-3456789")
    c.drawString(100, 580, "FINANCIAL SUMMARY & RISK METRICS:")
    c.drawString(100, 560, "- The entity shows flags of deteriorating capital reserves.")
    c.drawString(100, 540, "- High risk exposure detected due to severe unpaid supplier invoices.")
    c.drawString(100, 520, "- Active lawsuit filed by major vendor on 15/04/2026.")
    c.drawString(100, 500, "- Total outstanding liabilities exceed RM 450,000.")
    c.save()
    print(f"\u2705 Created {filename}")


def create_mock_invoice(filename="Sample_Invoice_INV-2026-0451.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "TAX INVOICE")
    c.drawString(100, 730, "------------------------------------------------------------")
    c.drawString(100, 710, "Invoice No: INV-2026-0451")
    c.drawString(100, 690, "Invoice Date: 12/08/2026")
    c.drawString(100, 670, "Due Date: 26/08/2026")
    c.drawString(100, 630, "Bill To: Meridian Trading Sdn Bhd")
    c.drawString(100, 610, "Attn: Siti Nurhaliza binti Ismail")
    c.drawString(100, 590, "Contact Email: siti.ismail@meridiantrading.com.my")
    c.drawString(100, 570, "Contact Phone: 019-8827364")
    c.drawString(100, 550, "Company Reg NRIC (Director): 850627-08-5142")
    c.drawString(100, 510, "ITEMS:")
    c.drawString(100, 495, "1. Industrial Packaging Rolls x 500 units - RM 8,500.00")
    c.drawString(100, 480, "2. Freight & Logistics Fee            - RM 620.00")
    c.drawString(100, 465, "3. Handling Surcharge                 - RM 150.00")
    c.drawString(100, 430, "Subtotal: RM 9,270.00")
    c.drawString(100, 415, "SST (6%): RM 556.20")
    c.drawString(100, 400, "TOTAL DUE: RM 9,826.20")
    c.drawString(100, 365, "Payment Status: UNPAID")
    c.drawString(100, 350, "Note: This is the third overdue reminder. Outstanding balance")
    c.drawString(100, 335, "has exceeded 45 days past due date. Risk of service suspension")
    c.drawString(100, 320, "if payment is not received within 7 days.")
    c.save()
    print(f"\u2705 Created {filename}")


def create_mock_po(filename="Sample_PO_PO-8842.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "PURCHASE ORDER")
    c.drawString(100, 730, "------------------------------------------------------------")
    c.drawString(100, 710, "PO Number: PO-8842")
    c.drawString(100, 690, "PO Date: 05/08/2026")
    c.drawString(100, 670, "Expected Delivery: 20/08/2026")
    c.drawString(100, 630, "Vendor: Apex Steelworks Sdn Bhd")
    c.drawString(100, 610, "Vendor Contact: Ahmad Faiz bin Zulkifli")
    c.drawString(100, 590, "Vendor Email: faiz.zulkifli@apexsteel.com.my")
    c.drawString(100, 570, "Vendor Phone: 012-5563981")
    c.drawString(100, 550, "Vendor Director NRIC: 780315-10-6231")
    c.drawString(100, 510, "ORDERED ITEMS:")
    c.drawString(100, 495, "1. Galvanized Steel Sheets 2mm x 200 units - RM 24,000.00")
    c.drawString(100, 480, "2. Mounting Brackets x 400 units           - RM 3,200.00")
    c.drawString(100, 465, "3. Delivery & Installation Service         - RM 1,500.00")
    c.drawString(100, 430, "TOTAL PO VALUE: RM 28,700.00")
    c.drawString(100, 395, "Payment Terms: Net 30")
    c.drawString(100, 380, "Vendor Risk Note: Vendor has an active lawsuit filed by a")
    c.drawString(100, 365, "former subcontractor regarding unpaid labour claims. Approve")
    c.drawString(100, 350, "with caution and require signed indemnity before deposit release.")
    c.save()
    print(f"\u2705 Created {filename}")


create_mock_credit_report()
create_mock_invoice()
create_mock_po()

# Pick which one to run through the pipeline below (change this to test a different file)
sample_file_path = "Sample_Invoice_INV-2026-0451.pdf"

✅ Created Credit_Assessment_Pinnacle.pdf
✅ Created Sample_Invoice_INV-2026-0451.pdf
✅ Created Sample_PO_PO-8842.pdf


### 4. (Option B) Upload Your Own Document
Run this cell instead of Step 3 to test with a real PDF or spreadsheet (`.xlsx` / `.csv`) — a credit report, invoice, or purchase order. Opens a native Colab file picker.

**If you already ran Step 3, running this too will overwrite `sample_file_path` with your uploaded file.**

In [4]:
from google.colab import files

print("\U0001F4E4 Please select a PDF or spreadsheet (.xlsx / .csv) file to upload...")
uploaded = files.upload()

if uploaded:
    sample_file_path = list(uploaded.keys())[0]
    print(f"\u2705 '{sample_file_path}' uploaded successfully.")
else:
    print("\u26A0\uFE0F No file uploaded \u2014 keeping the existing sample_file_path (if set).")

📤 Please select a PDF or spreadsheet (.xlsx / .csv) file to upload...


KeyboardInterrupt: 

### 5. Run the Extraction + Detection Pipeline
This runs `analyze_document()` on `sample_file_path`: extracts text, redacts PII, detects the document type, pulls structured fields for that type, and flags risk/exception lines.

Change `sample_file_path` above (Step 3 or 4) to test a different document, then re-run this cell.

In [8]:
result = analyze_document(sample_file_path)

--- FILE: Sample_Invoice_INV-2026-0451.pdf ---
Detected Document Type: INVOICE

--- PDPA SANITIZED OUTPUT ---
TAX INVOICE
------------------------------------------------------------
Invoice No: INV-2026-0451
Invoice Date: 12/08/2026
Due Date: 26/08/2026
Bill To: Meridian Trading Sdn Bhd
Attn: Siti Nurhaliza binti Ismail
Contact Email: [REDACTED_EMAIL]
Contact Phone: [REDACTED_PHONE]
Company Reg NRIC (Director): [REDACTED_NRIC]
ITEMS:
1. Industrial Packaging Rolls x 500 units - RM 8,500.00
2. Freight & Logistics Fee - RM 620.00
3. Handling Surcharge - RM 150.00
Subtotal: RM 9,270.00
SST (6%): RM 556.20
TOTAL DUE: RM 9,826.20
Payment Status: UNPAID
Note: This is the third overdue reminder. Outstanding balance
has exceeded 45 days past due date. Risk of service suspension
if payment is not received within 7 days.


--- STRUCTURED FIELDS ---
invoice_no: INV-2026-0451
invoice_date: 12/08/2026
due_date: 26/08/2026
bill_to: Meridian Trading Sdn Bhd
total_due: RM 9,826.20
payment_status: UNPA

### 6. AI Analysis Simulation + Dashboard Renderer

> ⚠️ **Note:** `simulate_llm_analysis()` below still returns a **hardcoded mock response** — it does not actually read the sanitized text or the structured fields from Step 5. This is fine for testing the pipeline/UI, but for a real submission you should replace it with a genuine LLM API call (see Step 8) so the risk score and summary actually reflect the uploaded document — and so it can speak to the specific document type detected (invoice vs. PO vs. credit report) rather than a generic credit-risk framing.

In [10]:
from IPython.display import display, HTML

def simulate_llm_analysis(sanitized_text: str, doc_type: str = "credit_report", fields: dict = None) -> dict:
    """
    Simulates sending the PDPA-compliant text (plus detected type and structured fields)
    to an LLM (e.g. Claude or Gemini) for a structured risk assessment.

    NOTE: This is a MOCK response for demo purposes — it does not vary based on the
    actual sanitized_text, doc_type, or fields passed in. Replace with a real API call
    (see Step 8) for production use.
    """
    prompt = f"""
    Context: You are a corporate financial risk analyst.
    Document type detected: {doc_type}
    Structured fields extracted: {fields}
    Task: Review the following sanitized {doc_type.replace('_', ' ')} and extract key insights.
    Format your response as a JSON object containing a Risk Score (1-100), a Summary,
    and a list of Actionable Recommendations.
    Data: {sanitized_text}
    """

    mock_llm_response = {
        "risk_score": 35,
        "risk_category": "Medium to High Risk",
        "executive_summary": "The document shows signs of financial distress or elevated commercial risk that warrants follow-up before further action is taken.",
        "actionable_recommendations": [
            "Request clarification on the flagged risk item before proceeding.",
            "Implement stricter payment or approval terms for this counterparty.",
            "Require additional documentation or guarantees before commitment."
        ],
        "explainability_flag": "Risk score weighted by the specific risk/exception lines detected in the document."
    }

    return mock_llm_response


def render_financial_dashboard(ai_insights: dict, doc_type: str = "", fields: dict = None):
    """Renders the AI insights (plus detected doc type and structured fields) as an HTML dashboard."""
    score_color = "red" if ai_insights["risk_score"] < 50 else "green"
    fields = fields or {}

    fields_html = "".join(
        f"<li><b>{k.replace('_', ' ').title()}:</b> {v if v else '<i>not found</i>'}</li>"
        for k, v in fields.items()
    ) or "<li><i>No structured fields available for this document type.</i></li>"

    dashboard_html = f"""
    <div style="font-family: sans-serif; border: 2px solid #e5e7eb; border-radius: 10px; padding: 20px; max-width: 800px;">
        <h2 style="color: #1f2937; border-bottom: 2px solid #e5e7eb; padding-bottom: 10px;">\U0001F4CA AI Financial Document Dashboard</h2>
        <p style="color: #6b7280; margin-top: -5px;">Detected Document Type: <b>{doc_type.replace('_', ' ').title() if doc_type else 'Unknown'}</b></p>

        <div style="display: flex; justify-content: space-between; margin-top: 20px;">
            <div style="background-color: #f9fafb; padding: 15px; border-radius: 8px; width: 45%;">
                <h3 style="margin: 0; color: #4b5563;">Risk Score</h3>
                <h1 style="margin: 10px 0; font-size: 48px; color: {score_color};">{ai_insights['risk_score']}/100</h1>
                <p style="margin: 0; font-weight: bold; color: {score_color};">{ai_insights['risk_category']}</p>
            </div>

            <div style="width: 50%;">
                <h3 style="margin-top: 0; color: #4b5563;">Executive Summary</h3>
                <p style="color: #374151; line-height: 1.5;">{ai_insights['executive_summary']}</p>
                <p style="font-size: 12px; color: #6b7280;"><i>Model Rationale: {ai_insights['explainability_flag']}</i></p>
            </div>
        </div>

        <h3 style="color: #4b5563; margin-top: 25px;">\U0001F4C4 Extracted Fields</h3>
        <ul style="color: #374151; line-height: 1.6;">
            {fields_html}
        </ul>

        <h3 style="color: #4b5563; margin-top: 25px;">\u26A1 Actionable Recommendations</h3>
        <ul style="color: #374151; line-height: 1.6;">
            {"".join(f"<li>{item}</li>" for item in ai_insights['actionable_recommendations'])}
        </ul>

        <p style="font-size: 11px; color: #9ca3af; margin-top: 20px; border-top: 1px solid #f3f4f6; padding-top: 10px;">
            \U0001F512 PDPA Notice: This document was processed in-memory only. Detected NRICs, emails, and phone numbers were redacted before analysis and are not stored or logged anywhere.
        </p>
    </div>
    """
    display(HTML(dashboard_html))

### 7. Run the Full Pipeline (Extraction + Dashboard)
This re-runs extraction on `sample_file_path`, then feeds the sanitized text, detected document type, and structured fields into the (currently mock) AI analysis step and renders the full dashboard.

In [9]:
result = analyze_document(sample_file_path)

print("\nSending sanitized data to AI Engine...\n")
ai_results = simulate_llm_analysis(
    result["sanitized_text"],
    doc_type=result["document_type"],
    fields=result["fields"],
)
render_financial_dashboard(ai_results, doc_type=result["document_type"], fields=result["fields"])

--- FILE: Sample_Invoice_INV-2026-0451.pdf ---
Detected Document Type: INVOICE

--- PDPA SANITIZED OUTPUT ---
TAX INVOICE
------------------------------------------------------------
Invoice No: INV-2026-0451
Invoice Date: 12/08/2026
Due Date: 26/08/2026
Bill To: Meridian Trading Sdn Bhd
Attn: Siti Nurhaliza binti Ismail
Contact Email: [REDACTED_EMAIL]
Contact Phone: [REDACTED_PHONE]
Company Reg NRIC (Director): [REDACTED_NRIC]
ITEMS:
1. Industrial Packaging Rolls x 500 units - RM 8,500.00
2. Freight & Logistics Fee - RM 620.00
3. Handling Surcharge - RM 150.00
Subtotal: RM 9,270.00
SST (6%): RM 556.20
TOTAL DUE: RM 9,826.20
Payment Status: UNPAID
Note: This is the third overdue reminder. Outstanding balance
has exceeded 45 days past due date. Risk of service suspension
if payment is not received within 7 days.


--- STRUCTURED FIELDS ---
invoice_no: INV-2026-0451
invoice_date: 12/08/2026
due_date: 26/08/2026
bill_to: Meridian Trading Sdn Bhd
total_due: RM 9,826.20
payment_status: UNPA

NameError: name 'simulate_llm_analysis' is not defined

### 8. (For Your Final Submission) Replace the Mock LLM Call

Right now `simulate_llm_analysis()` returns a **fixed dictionary** no matter what document, type, or fields are passed in. If judges upload a different invoice, PO, or credit report and see the same score and summary every time, that will hurt you on the "accuracy and quality of insights" success criterion.

To fix this, replace the body of `simulate_llm_analysis()` with a real API call that actually uses `doc_type` and `fields`, e.g.:

```python
import anthropic
import json

client = anthropic.Anthropic(api_key="YOUR_API_KEY")

def simulate_llm_analysis(sanitized_text: str, doc_type: str = "credit_report", fields: dict = None) -> dict:
    prompt = f'''
    Context: You are a corporate financial risk analyst.
    Document type: {doc_type}
    Structured fields already extracted: {fields}
    Task: Review the following sanitized {doc_type.replace('_', ' ')} and extract key insights,
    grounding your assessment in the structured fields and any risk/exception language present.
    Respond ONLY with a JSON object (no markdown, no preamble) with keys:
    risk_score (1-100 integer), risk_category (string), executive_summary (string),
    actionable_recommendations (list of strings), explainability_flag (string).
    Data: {sanitized_text}
    '''

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )

    raw_text = response.content[0].text.strip()
    raw_text = raw_text.replace("```json", "").replace("```", "").strip()
    return json.loads(raw_text)
```

You'd need to `!pip install anthropic` and store your API key securely (e.g. via Colab's Secrets manager, `google.colab.userdata`) rather than hardcoding it. Swap in Gemini's SDK instead if that's the model your team is using — the surrounding pipeline (extraction, masking, doc-type detection, field extraction, dashboard rendering) stays exactly the same either way.

**Also worth strengthening:** `detect_risks()` in Step 2 is a plain keyword match — it will only catch phrasing that literally contains words like "unpaid", "lawsuit", "overdue", etc. A real invoice using different wording ("payment overdue", "collections referral", "account delinquent") will slip through silently. If you have time, consider having the real LLM call in Step 8 do the risk/exception detection directly from `sanitized_text`, instead of relying on the keyword list — that alone would meaningfully improve accuracy for the "identify trends, patterns, exceptions" requirement.